# Review of TESS Sector 103 candidates

Companion to `reports/report_s103_<label>.md`. It loads the tables written by the pipeline, shows the
candidates that need a human look first (absent from catalogues, persistence not verifiable) and can
re-plot any candidate from the cleaned light curve.

Everything here is a **candidate for verification**, not a discovery.

Set `LABEL` to `"sample"` (development sample) or `"all"` (frozen final run).

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))  # come negli script: il .pth editable non è affidabile su iCloud
from vdxtess import config, plots, report

LABEL = "sample"
cfg = config.load()
SECTOR = cfg["dataset"]["sector"]
DATA = ROOT / "data"
tag = f"s{SECTOR:03d}_{LABEL}"
cand = pd.read_csv(DATA / f"crossmatch_{tag}.csv", keep_default_na=False, na_values=["", "nan", "NaN"])
counts = pd.read_csv(DATA / f"artifact_counts_{tag}.csv")
print(f"{len(cand)} candidates, sector {SECTOR}, {LABEL}")

## Artifact rejection, counts per criterion

In [ ]:
counts.assign(rejects_when=counts.criterion.map(report.CRITERIA))[
    ["criterion", "rejects_when", "all_sequential", "all_independent", "top50_sequential", "top50_independent"]]

## Catalogue labels and persistence

In [ ]:
display(pd.crosstab(cand.event.fillna("nessuno"), cand.catalog_label, margins=True))
display(pd.crosstab(cand.catalog_label, cand.persistence, margins=True))

## Candidates to look at first

In [ ]:
COLS = ["rank", "tic", "event", "tmag_hdr", "simbad_id", "simbad_otype", "vsx_type", "catalog_label",
        "persistence", "persistence_detail", "variabile_vicina", "warnings", "drivers"]
first = cand[(cand.catalog_label == "assente") | (cand.persistence == "persistenza non verificabile")]
first[COLS]

Plots written by the pipeline (copied next to the report by `scripts/fase6.py`).

In [ ]:
fig_dir = ROOT / "reports" / "figures" / tag
for tic in first.tic.astype(int):
    found = sorted(fig_dir.glob(f"*_tic{tic}.png"))
    display(Markdown(f"### TIC {tic}"))
    display(Image(filename=str(found[0])) if found else Markdown("_plot missing: run scripts/fase6.py_"))

## Re-plot one candidate from the cleaned light curve

Needs `data/cache/clean/` (not in the repository: rebuild it with `scripts/fase1.py`).

In [ ]:
TIC = int(first.tic.iloc[0])
row = cand.loc[cand.tic == TIC].iloc[0]
clean = DATA / "cache" / "clean" / f"s{SECTOR:04d}" / f"tic{TIC}.parquet"
if clean.exists():
    out = DATA / "plots" / "notebook" / f"tic{TIC}.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    plots.candidate(pd.read_parquet(clean), row, cfg["features"], out,
                    f"rank {row['rank']}  TIC {TIC}  Tmag {row.tmag_hdr:.2f}  sector {SECTOR}\n{row.drivers}")
    display(Image(filename=str(out)))
else:
    print(f"missing {clean.relative_to(ROOT)}")